# Mini Projeto 05

# Análise de Churn - Modelagem Estatística e Interpretação de Resultados

---
## 01 - Importação das Bibliotecas

In [1]:
# imports
import pandas as pd
import numpy as np
import statsmodels as sm
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

In [2]:
# configuracoes para melhor visualizacao
pd.set_option("display.float_format", lambda x: '%.4f' % x)

In [3]:
%reload_ext watermark
%watermark -a "Wolfdata"

Author: Wolfdata



In [4]:
%watermark --iversions

IPython    : 9.13.0
matplotlib : 3.10.9
numpy      : 2.4.4
pandas     : 3.0.2
plotly     : 6.7.0
seaborn    : 0.13.2
statsmodels: 0.14.6



---
## 02 - Definição do Problema de Negócio

In [5]:
display(Markdown(""" 
### 02.1 - Problema de Negócio
                 
A **Connecta Telecon**, uma empresa fictícia de telecomunicações, está enfrentando uma taxa de cancelamento de serviços (churn) acima da média do setor.
                 
A perda de clientes não só impacta a receita recorrente, más também gera custos elevados com a aquisição de novos clientes para substituir os que foram perdidos.
                                 
A diretoria precisa de respostas claras e baseada em dados para a seguinte pergunta:
                 
**"Quais são os principais fatores que levam nossos clientes a cancelar o serviço?"**                 

- Queremos compreender a relação entre variáveis -> Modelagem Estatística
- Se quisermos prever o cancelamento (churn) -> Modelagem Predidita

### 02.2 - Objetivos do Projeto

1. **Identificar os Fatores-Chave:** Determinar quais variáveis (como tipo de contrato, tempo de fidelidade, valor da fatura) têm um impacto estatisticamente 
siginificativo na probabilidade de um cliente cancelar o serviço.

2. **Quantificaar o Impacto:** Medir o quão forte é a influência de cada fator no risco de churn.

3. **Gerar Recomentações:** Traduzir os resultados da análise estatística em recomendações de negócio acionáveis para a criação de estratégias de retenção de clientes.

O modelo escolhido apra esta análise será a **Regressão Logística**, pois o nosso objetivo é entender a relação entre diversas variáveis e uma variável de resultado binária (Churn: Sim ou Não).

"""))

 
### 02.1 - Problema de Negócio

A **Connecta Telecon**, uma empresa fictícia de telecomunicações, está enfrentando uma taxa de cancelamento de serviços (churn) acima da média do setor.

A perda de clientes não só impacta a receita recorrente, más também gera custos elevados com a aquisição de novos clientes para substituir os que foram perdidos.

A diretoria precisa de respostas claras e baseada em dados para a seguinte pergunta:

**"Quais são os principais fatores que levam nossos clientes a cancelar o serviço?"**                 

- Queremos compreender a relação entre variáveis -> Modelagem Estatística
- Se quisermos prever o cancelamento (churn) -> Modelagem Predidita

### 02.2 - Objetivos do Projeto

1. **Identificar os Fatores-Chave:** Determinar quais variáveis (como tipo de contrato, tempo de fidelidade, valor da fatura) têm um impacto estatisticamente 
siginificativo na probabilidade de um cliente cancelar o serviço.

2. **Quantificaar o Impacto:** Medir o quão forte é a influência de cada fator no risco de churn.

3. **Gerar Recomentações:** Traduzir os resultados da análise estatística em recomendações de negócio acionáveis para a criação de estratégias de retenção de clientes.

O modelo escolhido apra esta análise será a **Regressão Logística**, pois o nosso objetivo é entender a relação entre diversas variáveis e uma variável de resultado binária (Churn: Sim ou Não).



---
## 03 - Extração dos Dados

In [10]:
# funcao para gerar dados ficticios mas coerentes para analise
def gerar_dados_churn(numero_clientes: int = 2000) -> pd.DataFrame:

    # reprodutibilidade
    np.random.seed(42)

    # variaveis
    fidelidade_meses = np.random.randint(1, 73, size = numero_clientes)

    tipo_contrato_opts = ["Mensal", "Anual", "Dois anos"]
    contrato_probs = [0.6, 0.25, 0.15]
    tipo_contrato = np.random.choice(tipo_contrato_opts, size = numero_clientes, p = contrato_probs)

    servico_internet_opts = ["Fibra Óptica", "DSL", "Não"]
    internet_probs = [0.55, 0.35, 0.10]
    servico_internet = np.random.choice(servico_internet_opts, size = numero_clientes, p = internet_probs)

    fatura_base = {
        "Mensal": np.random.normal(60, 20),
        "Anual": np.random.normal(70, 25),
        "Dois anos": np.random.normal(80, 25)
    }

    fatura_mensal = [fatura_base[c] + fidelidade_meses[i] * 0.2 + np.random.normal(0, 5) for i, c in enumerate(tipo_contrato)]
    fatura_mensal = np.clip(fatura_mensal, 20, 120)

    # logica para a probabilidade de churn
    # clientes com contrato mensal, baixa fidelidade e fatura alta tem maior chance de churn
    prob_churn_log = -2.5  # Intercepto base (tendência a não cancelar)
    prob_churn_log += -0.05 * fidelidade_meses  # Mais fidelidade, menor chance
    prob_churn_log += [3.0 if c == 'Mensal' else -1.5 if c == 'Anual' else -2.5 for c in tipo_contrato] # Contrato mensal aumenta muito a chance
    prob_churn_log += [0.8 if s == 'Fibra Óptica' else -0.5 for s in servico_internet] # Fibra tende a ter mais churn (talvez por preço)
    prob_churn_log += 0.03 * fatura_mensal # Fatura mais alta, mais chance

    # Converter log-odds para probabilidade usando a função sigmoide
    prob_churn = 1 / (1 + np.exp(-prob_churn_log))

    # Gerar o resultado de churn com base na probabilidade
    churn = np.random.binomial(1, prob_churn)


    df = pd.DataFrame({
        'ID_Cliente': range(1, numero_clientes + 1),
        'Fidelidade_Meses': fidelidade_meses,
        'Tipo_Contrato': tipo_contrato,
        'Servico_Internet': servico_internet,
        'Fatura_Mensal': fatura_mensal,
        'Churn': churn
    })

    return df

In [11]:
# gerar os dados
df_churn = gerar_dados_churn()

In [13]:
display(Markdown("### Amostra de Dados Gerados"))
df_churn.head()

### Amostra de Dados Gerados

,ID_Cliente,Fidelidade_Meses,Tipo_Contrato,Servico_Internet,Fatura_Mensal,Churn
0,1,52,Anual,Fibra Óptica,36.5724,0
1,2,15,Dois anos,Fibra Óptica,45.2871,0
2,3,72,Mensal,Fibra Óptica,101.0033,1
3,4,61,Mensal,Fibra Óptica,103.3314,1
4,5,21,Mensal,DSL,94.0969,1


---
## 04 - Análise Exploratória de Dados (EDA)

In [14]:
display(Markdown("### Informções Gerais do DataFrame"))
df_churn.info()

### Informções Gerais do DataFrame

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID_Cliente        2000 non-null   int64  
 1   Fidelidade_Meses  2000 non-null   int64  
 2   Tipo_Contrato     2000 non-null   str    
 3   Servico_Internet  2000 non-null   str    
 4   Fatura_Mensal     2000 non-null   float64
 5   Churn             2000 non-null   int64  
dtypes: float64(1), int64(3), str(2)
memory usage: 122.8 KB
